In [1]:
!pwd

/home/alan_khang/dev/mirror3d


In [2]:
import os
import sys
import json
import numpy as np
from glob import glob
import cv2
from sklearn.cluster import KMeans
from pycocotools.coco import COCO

In [3]:
data_dir = './data/Mirror-Glass_Segmentation.v11i.coco-segmentation'

**Generate 10 means normal from training mirror planes**

In [4]:
data_split = 'train'

color_imgs = glob(f'{data_dir}/{data_split}/*.jpg')

error_pcd_file = f'{data_dir}/anno_progress/{data_split}/error_pcd_list.txt'

with open(error_pcd_file, 'r') as f:
    error_pcd_list = f.read().splitlines()

error_pcd_filenames = [os.path.basename(pcd_fn).split('_idx')[0] for pcd_fn in error_pcd_list]

mirror_normal_list = []
skipping_count = 0

for i, color_img in enumerate(color_imgs):
    color_img_fn = color_img

    color_img_name = os.path.basename(color_img_fn)
    mirror_plane_fp = data_dir + f'/output/{data_split}/mirror_plane/' + color_img_name.replace('.jpg', '.json')

    img_fn = f"{color_img_name.split('.')[0]}"
    if img_fn in error_pcd_filenames:
        print(f"Skipping {img_fn} due to error in point cloud generation.")
        skipping_count += 1
        continue

    mirror_plane_json = json.load(open(mirror_plane_fp, 'r'))
    for plane in mirror_plane_json:
        mirror_normal_list.append(plane['normal'])

Skipping oak_left_frame_0000263_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000151_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000127_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_right_frame_0000486_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000129_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_right_frame_0000209_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_right_frame_0000459_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_front_frame_0000405_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000150_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000133_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000358_2026

In [5]:
skipping_count

62

In [6]:
len(mirror_normal_list)

458

In [7]:
mirror_normal_list = np.array(mirror_normal_list)
kmeans = KMeans(n_clusters=10)
labels = kmeans.fit_predict(mirror_normal_list)

mean_normal_vecs = kmeans.cluster_centers_

In [8]:
mean_normal_fp = data_dir + '/kmeans_normal_10.npy'
np.save(mean_normal_fp, mean_normal_vecs)

**Generate json annotation file for training data**

In [9]:
mean_normals_fp = data_dir + '/kmeans_normal_10.npy'
mean_normal_vecs = np.load(mean_normals_fp)

In [10]:
data_splits = ['train', 'valid']

for data_split in data_splits:
    error_pcd_file = f'{data_dir}/anno_progress/{data_split}/error_pcd_list.txt'

    with open(error_pcd_file, 'r') as f:
        error_pcd_list = f.read().splitlines()

    error_pcd_filenames = [os.path.basename(pcd_fn).split('_idx')[0] for pcd_fn in error_pcd_list]

    annot_fp = data_dir + f'/{data_split}_annot.json'

    semg_coco_json = data_dir + f'/{data_split}/_annotations.coco.json'
    semg_coco = COCO(semg_coco_json)

    image_list = []
    annot_list = []
    categories =[{"supercategory":"mirror_glass", "id":1, "name":"mirror_glass"}]

    correct_img_id = 1
    correct_annot_id = 1

    for img_obj in semg_coco.imgs.values():
        img_id = img_obj['id']
        img_fn = img_obj['file_name']
        img_w, img_h = img_obj['width'], img_obj['height']

        img_name = f"{img_fn.split('.')[0]}"
        if img_name in error_pcd_filenames:
            print(f"Skipping {img_name} due to error in point cloud generation.")
            continue

        new_img_id = correct_img_id
        mirror_color_img_fp = f"{data_split}/{img_fn}"
        raw_sensorD_fp = f"depth/{data_split}/{img_fn.replace('.jpg', '.png')}"
        refined_sensorD_fp = f"output/{data_split}/anno_result/{img_fn.replace('.jpg', '.png')}"
        mirror_inst_mask_fp = f'int_mask/{data_split}/{img_fn.replace(".jpg", ".png")}'
        mirror_plane_fp = data_dir + f'/output/{data_split}/mirror_plane/' + img_fn.replace('.jpg', '.json')

        mirror_plane_json = json.load(open(mirror_plane_fp, 'r'))

        segm_annot_ids = semg_coco.getAnnIds(imgIds=img_id)
        segm_anns = semg_coco.loadAnns(segm_annot_ids)

        assert len(segm_anns) == len(mirror_plane_json), f"#segm. annot. ({len(segm_anns)}) != #mirror planes ({len(mirror_plane_json)}) for image {img_fn}"

        for annot_idx, (segm_ann, mirror_plane) in enumerate(zip(segm_anns, mirror_plane_json)):
            new_annot_id = correct_annot_id
            area = segm_ann['area']
            bbox = segm_ann['bbox']
            segmentation = segm_ann['segmentation']
            iscrowd = segm_ann['iscrowd']

            plane_normal = mirror_plane['normal']
            plane_normal_arr = np.array(plane_normal)
            inst_tag = mirror_plane['mask_id']
            assert annot_idx + 1 == inst_tag, f"Annotation index {annot_idx} does not match instance tag {inst_tag} for image {img_fn}"

            anchor_normal_class = np.argmin(np.linalg.norm(plane_normal_arr - mean_normal_vecs, axis=1))
            anchor_normal_residual = plane_normal_arr - mean_normal_vecs[anchor_normal_class]

            annot_list.append({
                'id': new_annot_id,
                'image_id': new_img_id,
                'category_id': 1,
                'iscrowd': iscrowd,
                'area': area,
                'bbox': bbox,
                'segmentation': segmentation,
                'width': img_w,
                'height': img_h,
                'mirror_normal_camera': plane_normal,
                'anchor_normal_class': int(anchor_normal_class),
                'anchor_normal_residual': anchor_normal_residual.tolist(),
                'raw_sensorD_path': raw_sensorD_fp,
                'refined_sensorD_path': refined_sensorD_fp,
                'raw_meshD_path': raw_sensorD_fp,
                'refined_meshD_path': refined_sensorD_fp,
                'mirror_color_image_path': mirror_color_img_fp,
                'file_name': mirror_color_img_fp,
                'instance_tag': int(inst_tag),
                'mirror_instance_mask_path': mirror_inst_mask_fp
            })

            correct_annot_id += 1

        image_list.append({
            'id': new_img_id,
            'height': img_h,
            'width': img_w,
            'mirror_color_image_path': mirror_color_img_fp,
            'raw_sensorD_path': raw_sensorD_fp,
            'refined_sensorD_path': refined_sensorD_fp,
            'raw_meshD_path': raw_sensorD_fp,
            'refined_meshD_path': refined_sensorD_fp,
            'mirror_instance_mask_path': mirror_inst_mask_fp
        })

        correct_img_id += 1

    json_output = {'annotations': annot_list, 'images': image_list, 'categories': categories}

    with open(annot_fp, 'w') as f:
        json.dump(json_output, f)

loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Skipping oak_left_frame_0000149_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_right_frame_0000241_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_front_frame_0000181_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_right_frame_0000086_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_front_frame_0000407_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_right_frame_0000254_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_front_frame_0000142_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_front_frame_0000483_2026_03_23-20-44-59_jpg due to error in point cloud generation.
Skipping oak_front_frame_0000180_2026_03_23-20-56-12_jpg due to error in point cloud generation.
Skipping oak_left_frame_0000150_2026_03_23-20